# Task 4: Tracking individual eggs in a video

Video: [Raw and Boiled Egg Spinning](https://www.youtube.com/watch?v=Y4JyV94y2_A) (60 s, 24 fps, 640×360).

**Method** (code: `src/track_video.py` and `src/classical_seg.py` in the repo):
1. **Detection:** the final classical algorithm from Task 2, adapted for video (`configs/video_v1.yaml`):
   * Smaller size limits, since the eggs cover about 0.5% of a frame vs 2.6–38% in the dataset.
   * No forced egg, so the title cards stay empty.
2. **Tracking (SORT-style):**
   * A Kalman filter per egg predicts where it will be in the next frame.
   * Hungarian matching on distance and box overlap assigns detections to eggs.
   * A track is confirmed after 3 hits in a row, so one-frame blips get no ID.
   * A hidden egg (covered, or held in the hand) keeps its track. Its estimate is moved by the optical flow of
     whatever covers it, so it is carried along with the hand, and it locks onto the visible white part of the
     held egg. Its label stays at that estimated position.
   * When the egg is put down, it is recovered by its propagated estimate, so it keeps its ID.

No GPU is needed: everything is classical image processing and runs on the CPU cores.

### Kaggle settings
* **Settings → Internet → On** (to clone the repo and download the video).
* **Accelerator: None** is fine. CPU sessions have 4 cores, and the detection step uses all of them.

In [ ]:
!pip install -q -U yt-dlp
import os, subprocess
from pathlib import Path

REPO = Path('/kaggle/working/Egg_Segmentation')
if (REPO / '.git').exists():      # re-running the notebook: update to the latest commit
    !git -C {REPO} fetch -q origin main && git -C {REPO} reset -q --hard origin/main
else:
    !git clone -q --depth 1 https://github.com/Duggineniakhil/Egg_Segmentation.git {REPO}
%cd {REPO}
!git log --oneline -1
assert (REPO / 'src' / 'track_video.py').exists(), 'src/track_video.py missing: push the Task 4 commit to GitHub first'

## Get the video
The cell uses an `.mp4` attached to the notebook (**Add Input → Upload → New Dataset**) if there is one. Otherwise it
downloads the video from YouTube with yt-dlp. YouTube sometimes blocks downloads from cloud servers ("Sign in to
confirm you're not a bot"); if that happens, attach the video as a dataset and re-run.

Any resolution works. Detection runs on frames scaled down to 640 px wide, because the algorithm's pixel parameters
are tuned for that size, and the output video is drawn at the original resolution.

In [ ]:
import glob
import cv2

URL = 'https://www.youtube.com/watch?v=Y4JyV94y2_A'
attached = sorted(glob.glob('/kaggle/input/**/*.mp4', recursive=True))
if attached:
    VIDEO = Path(attached[0])
else:
    VIDEO = Path('/kaggle/working/egg_video.mp4')
    if not VIDEO.exists():
        r = subprocess.run(['yt-dlp', '-q', '--no-warnings', '-f', '18/best[ext=mp4][height<=1080]/best',
                            '-o', str(VIDEO), URL], capture_output=True, text=True)
        if r.returncode != 0 or not VIDEO.exists():
            raise SystemExit('yt-dlp could not download the video (' + r.stderr.strip()[-300:] +
                             '). Attach the video as a Kaggle dataset and re-run this cell.')

cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened(), f'cannot open {VIDEO}'
print(VIDEO, f'{int(cap.get(cv2.CAP_PROP_FRAME_COUNT))} frames, {cap.get(cv2.CAP_PROP_FPS):.0f} fps, '
      f'{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}')
cap.release()

## Run detection and tracking
Detection runs in parallel on all CPU cores, which takes roughly 10–15 minutes for the full video. Tracking itself takes a few seconds.

In [ ]:
OUT = Path('/kaggle/working/tracking')
CACHE = Path('/kaggle/working/dets.pkl')
!python src/track_video.py --video "{VIDEO}" --config configs/video_v1.yaml --out "{OUT}" --workers {os.cpu_count()} --cache "{CACHE}"

## Check the IDs
`src/check_tracks.py` compares the tracks with identity checkpoints written down by watching the video frame by
frame. Egg A (ID 1) is the raw egg: picked up, spun on the right, then held and put down at the far left. Egg B
(ID 2) is taken by the same hand at about 30 s, carried right and spun there. The previous tracker version
swapped the two IDs at that handover and fails the last two checks.

In [ ]:
scale = cv2.VideoCapture(str(VIDEO)).get(cv2.CAP_PROP_FRAME_WIDTH) / 640
!python src/check_tracks.py "{OUT}/tracks.csv" --scale {scale}
import pandas as pd
log = pd.read_csv(OUT / 'tracking_log.csv')
print(log[log.event.isin(['new_id', 'recover', 'retire'])].to_string(index=False))

## Convert to H.264
OpenCV writes MP4V, which many browsers and Google Drive's previewer can't play. ffmpeg re-encodes the video to H.264.

In [ ]:
!ffmpeg -y -loglevel error -i "{OUT}/tracked_raw.mp4" -c:v libx264 -pix_fmt yuv420p -crf 23 -movflags +faststart "{OUT}/tracked_eggs.mp4"
!ls -la "{OUT}"

## Results

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

summary = json.loads((OUT / 'summary.json').read_text())
print('track ids:', summary['track_ids'])
print(pd.DataFrame(summary['tracks']).T)

# preview: 12 frames from the part of the video with eggs
tracks = pd.read_csv(OUT / 'tracks.csv')
f0, f1 = tracks.frame.min(), tracks.frame.max()
cap = cv2.VideoCapture(str(OUT / 'tracked_eggs.mp4'))
tiles = []
for f in np.linspace(f0, f1, 12).astype(int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, im = cap.read()
    if ok:
        tiles.append(cv2.cvtColor(cv2.resize(im, (480, 270)), cv2.COLOR_BGR2RGB))
sheet = np.vstack([np.hstack(tiles[i:i + 3]) for i in range(0, len(tiles) - len(tiles) % 3, 3)])
cv2.imwrite(str(OUT / 'preview_sheet.jpg'), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))
plt.figure(figsize=(18, 18 * sheet.shape[0] / sheet.shape[1])); plt.imshow(sheet); plt.axis('off'); plt.show()

# trajectories of each egg over time
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
for tid, g in tracks.groupby('id'):
    ax[0].plot(g.frame / summary['fps'], g.cx, label=f'Egg {tid}')
    ax[1].plot(g.frame / summary['fps'], g.cy, label=f'Egg {tid}')
for a, l in zip(ax, ['x (px)', 'y (px)']):
    a.set_xlabel('time (s)'); a.set_ylabel(l); a.legend(frameon=False)
plt.tight_layout(); plt.show()

## Debug frames
Annotated frames from the hand pick-ups and the 30 s handover (`tracking/debug_frames/`).

In [ ]:
dbg = sorted((OUT / 'debug_frames').glob('*.jpg'))
fig, axes = plt.subplots((len(dbg) + 2) // 3, 3, figsize=(18, 3.6 * ((len(dbg) + 2) // 3)))
for ax in axes.ravel():
    ax.axis('off')
for ax, p in zip(axes.ravel(), dbg):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)); ax.set_title(p.stem, fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# embed the video in the notebook if it is small enough (a full-HD file can be large)
from IPython.display import Video
size_mb = (OUT / 'tracked_eggs.mp4').stat().st_size / 1e6
print(f'tracked_eggs.mp4: {size_mb:.1f} MB')
Video(str(OUT / 'tracked_eggs.mp4'), embed=True, width=640) if size_mb < 40 else print('too large to embed: download it below')

## Download the outputs
Click the link to download `tracking_outputs.zip`. It contains:
* `tracked_eggs.mp4`: the annotated video.
* `tracks.csv`: one row per egg per frame.
* `summary.json`, `tracking_log.csv` (new IDs, misses, recoveries, retirements) and `debug_frames/`
* `preview_sheet.jpg`

Upload `tracked_eggs.mp4` to Google Drive, set **Share → Anyone with the link → Viewer**, and put that link in the report.

In [ ]:
import shutil
from IPython.display import FileLink
for f in ['tracked_raw.mp4']:
    (OUT / f).unlink(missing_ok=True)      # keep only the H.264 version
shutil.make_archive('/kaggle/working/tracking_outputs', 'zip', OUT)
os.chdir('/kaggle/working')
FileLink('tracking_outputs.zip')